# Aquarium motion model

An original browser-Python reinterpretation inspired by **Reference-Image Aquarium Game**, credited in the Awesome GPT-6 Astra catalog to Tim Jayas.

Sources: [original creator post](https://x.com/TimJayas/status/2095611134992945385) · [Awesome GPT-6 Astra](https://github.com/magiccreator-ai/awesome-gpt-6-astra#reference-image-aquarium-game)

> This notebook does not reproduce the creator's code, assets, prompt, or benchmark. It demonstrates a small bounded-motion model with reproducible Python.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(6)
n = 18
pos = rng.uniform([0, 0], [10, 6], size=(n, 2))
vel = rng.normal(0, 0.12, size=(n, 2))
trail = []

for _ in range(180):
    # Gentle schooling: drift toward the group centroid plus random steering.
    centroid = pos.mean(axis=0)
    vel += 0.008 * (centroid - pos) + rng.normal(0, 0.015, size=vel.shape)
    speed = np.linalg.norm(vel, axis=1, keepdims=True)
    vel = vel / np.maximum(speed, 1e-9) * np.minimum(speed, 0.18)
    pos += vel
    for axis, bound in [(0, 10), (1, 6)]:
        low = pos[:, axis] < 0
        high = pos[:, axis] > bound
        vel[low | high, axis] *= -1
        pos[:, axis] = np.clip(pos[:, axis], 0, bound)
    trail.append(pos.copy())

trail = np.array(trail)
fig, ax = plt.subplots(figsize=(9, 5))
for i in range(n):
    ax.plot(trail[:, i, 0], trail[:, i, 1], alpha=.35)
ax.scatter(pos[:, 0], pos[:, 1], s=45)
ax.set(xlim=(0,10), ylim=(0,6), xlabel='x', ylabel='y', title='Bounded schooling trajectories')
plt.show()

## Try it
Change `n`, the steering coefficient, or maximum speed. A richer version could add obstacles, predator/prey rules, or depth as a third coordinate.